# Breach targets, transaction costs and turnover

Evaluate sensitivity to the breach target and transaction costs, and measure
hedging turnover. The default test regime is 2022. Run after the controller
comparison has produced its policy and episode records.

Tail-probability and bootstrap block-length analyses are implemented in
`new-reward/robust_reanalysis.py`.


## Environment


In [ ]:
# CTC local runtime
from pathlib import Path
import os, sys, json, time, math, random, glob, hashlib, importlib, shutil, copy
from itertools import product
from scipy import stats
from collections import deque
def _find_repository_root():
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / 'ctc_runtime.py').is_file():
            return candidate
    raise FileNotFoundError('Start Jupyter from the conformal-trust-control repository.')

KIT_ROOT = _find_repository_root()
if str(KIT_ROOT) not in sys.path:
    sys.path.insert(0, str(KIT_ROOT))
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from scipy.stats import norm
from IPython.display import display
from ctc_runtime import initialize, generate_rollout

globals().update(initialize())
os.chdir(LOCAL_REPO)
DEVICE = torch.device('cpu')
torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)
print('Repository:', KIT_ROOT)
print('Device:', DEVICE, '| torch:', torch.__version__, '| threads:', torch.get_num_threads())


## Train, validation and test years


In [ ]:
# ── REGIME config (mixed multi-year training version) ──
#  mixed training: TRAIN_YEARS can be"long and diverse"year list(including bull/bear/crash),
#  so g1 learns to control the tail across all regimes. But it must strictly satisfy: every train year < valyear < testyear.

# ── preset: uncomment one of these lines (crash panel: run first crash_test2018, then change to crash_test2020 rerun the fullnotebook) ──
REGIME, TRAIN_YEARS, VAL_YEARS, TEST_YEARS = 'crash_test2018', list(range(2008,2017)), [2017], [2018]    # clean OOS crash: Feb Volmageddon + Q4 sharp drop
# REGIME, TRAIN_YEARS, VAL_YEARS, TEST_YEARS = 'crash_test2020', list(range(2008,2019)), [2019], [2020]   # clean OOS crash: COVID (full year + crash slice)
# REGIME, TRAIN_YEARS, VAL_YEARS, TEST_YEARS = 'mix_test2022', list(range(2008,2021)), [2021], [2022]      # mixed 13 years → bear market test
# REGIME, TRAIN_YEARS, VAL_YEARS, TEST_YEARS = 'mix_test2024', list(range(2008,2023)), [2023], [2024]      # mixed → bull market test
# REGIME, TRAIN_YEARS, VAL_YEARS, TEST_YEARS = 'narrow_test2022', [2019,2020], [2021], [2022]              # narrow-training control

# ── strictly guard against look-ahead (hard assertions): every train year < every val year < every test year ──
assert max(TRAIN_YEARS) < min(VAL_YEARS),  f'LOOK-AHEAD! train latest year {max(TRAIN_YEARS)} no earlier than val {min(VAL_YEARS)}'
assert max(VAL_YEARS)   < min(TEST_YEARS), f'LOOK-AHEAD! val latest year {max(VAL_YEARS)} no earlier than test {min(TEST_YEARS)}'
assert not (set(TRAIN_YEARS) | set(VAL_YEARS)) & set(TEST_YEARS), 'test year must not appear in train/val'

# ── crash sub-window: in"full year test", and within it separately evaluate the true crash period (verdict cell use) ──
#    decision rule: episode live interval [date_start, date_end] with any crash window"overlaps"counts it in,
#    because if the holding period touches the crash at all, this hedge has lived through the crash.
CRASH_WINDOWS = {
    'crash_test2018': [('2018-02-01', '2018-02-12'),   # Volmageddon (XIV blow-up, VIX doubled)
                       ('2018-10-01', '2018-12-26')],  # Q4 selloff all the way to the Christmas low
    'crash_test2020': [('2020-02-19', '2020-04-07')],  # COVID crash (top -> 3/23 bottom -> initial rebound)
}
CRASH_SLICES = CRASH_WINDOWS.get(REGIME, [])

OUT_DIR = f'{DRIVE_REPO}/checkpoints/regime_{REGIME}'
os.makedirs(OUT_DIR, exist_ok=True)
assert os.path.exists(CLEAN_PARQUET), f'missing parquet: {CLEAN_PARQUET}'
print(f'REGIME={REGIME}')
print(f'  TRAIN={list(TRAIN_YEARS)}  (all < val)')
print(f'  VAL  ={VAL_YEARS}  (< test)')
print(f'  TEST ={TEST_YEARS}')
if CRASH_SLICES: print(f'  CRASH sub-slice={CRASH_SLICES}')
print('✓ strictly forward: max(train) < min(val) <= max(val) < min(test) — no look-ahead')


## Construct the data splits


In [ ]:
# ── split years by REGIME train/val/test ──
df_all = pd.read_parquet(CLEAN_PARQUET)
df_all['date'] = pd.to_datetime(df_all['date'])
yr = df_all['date'].dt.year
OUT_COLS = ['quote_datetime','ticker','expiration','strike',
            'underlying_bid','underlying_ask','bid','ask','nbr_next_steps']
for name, years in [('train', TRAIN_YEARS), ('validation', VAL_YEARS), ('test', TEST_YEARS)]:
    sub = df_all.loc[yr.isin(years)].sort_values(['expiration','strike','date']).reset_index(drop=True)
    g = sub.groupby(['expiration','strike'], sort=False)
    sub['nbr_next_steps'] = (g['date'].transform('count') - g.cumcount() - 1).astype('int32')
    sub[OUT_COLS].to_csv(f'data/{name}.csv', index=False)
    print(f'  {name:11s}: years={years}  rows={len(sub):>9,}')
rates = (df_all[['date','r']].drop_duplicates('date').sort_values('date')
         .rename(columns={'date':'Date','r':'1y'}))
rates['Date'] = pd.to_datetime(rates['Date']).dt.strftime('%Y-%m-%d')
old = pd.read_csv('data/1yr_treasury.csv') if os.path.exists('data/1yr_treasury.csv') else pd.DataFrame()
rates = pd.concat([old, rates]).drop_duplicates('Date').sort_values('Date')
rates.to_csv('data/1yr_treasury.csv', index=False)
print('treasury OK')


## Black–Scholes Greeks


In [ ]:
from scipy.stats import norm

def bs_greeks_inner(S, K, r, q, vol, T_years):
    if T_years <= 0 or vol <= 1e-8 or S <= 0 or K <= 0:
        return 0.0, 0.0, 0.0, 0.0
    sqT = np.sqrt(T_years)
    d1 = (np.log(S/K) + (r - q + 0.5*vol*vol) * T_years) / (vol * sqT)
    d2 = d1 - vol * sqT
    n_d1 = norm.pdf(d1)
    return (float(norm.cdf(d1)),
            float(n_d1 / (S * vol * sqT)),
            float(S * n_d1 * sqT),
            float(-(S * n_d1 * vol) / (2 * sqT) - r * K * np.exp(-r * T_years) * norm.cdf(d2)))


## Evaluation functions


In [ ]:
STATE_DIM, ACTION_DIM = 8, 1
TC_RATE = 0.001
REWARD_EXP = 1.0
GAMMA = 0.99
Q_DIV = 0.0
LAMBDAS_DEC = [0.0, 0.25, 0.5, 0.75, 1.0]
RATES = pd.read_csv('data/1yr_treasury.csv')
RATES['Date'] = pd.to_datetime(RATES['Date'])

def get_r(date):
    rows = RATES[RATES['Date']==date]
    if len(rows): return float(rows['1y'].iloc[0])
    rows = RATES[RATES['Date']<=date].sort_values('Date')
    return float(rows['1y'].iloc[-1]) if len(rows) else 0.04

def build_state(row, position_before_action):
    T_days = float(row['T'])
    spot   = float(row['S-1'])
    K      = spot / float(row['S/K'])
    IV     = float(row['v'])
    date   = pd.to_datetime(row['Date'])
    r      = get_r(date)
    d, g, vega, theta = bs_greeks_inner(spot, K, r, Q_DIV, IV, max(T_days, 0)/365.0)
    return np.array([float(row['S/K']), T_days/30.0, position_before_action, IV,
                     d, g, vega, theta], dtype=np.float32)

def cvar_5(arr):
    arr = np.asarray(arr); n_tail = max(1, int(0.05 * len(arr)))
    return np.sort(arr)[:n_tail].mean()


## Generate episode paths


In [ ]:
SEARCH_KAPPA = 2
REP_NAME = 'bs_reference'

def _csv_years(path):
    return set(pd.to_datetime(pd.read_csv(path, usecols=['Date'])['Date']).dt.year.astype(int))

def _file_md5(p, chunk=1<<20):
    h = hashlib.md5()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''): h.update(b)
    return h.hexdigest()[:12]

def gen_rollout(split):
    return generate_rollout(split, REGIME, LOCAL_REPO)

ROLLF = {split: gen_rollout(split) for split in ('train', 'validation', 'test')}

def _episodes_from_csv(p):
    csv = pd.read_csv(p); csv['Date'] = pd.to_datetime(csv['Date'])
    eps = []
    for eid, edf in csv.groupby('episode', sort=False):
        edf = edf.sort_values('gstep').reset_index(drop=True)
        eps.append({'episode': int(eid), 'date_start': edf['Date'].iloc[0], 'ep_step_df': edf})
    return pd.DataFrame(eps).sort_values('date_start').reset_index(drop=True)

def load_eps(split): return _episodes_from_csv(ROLLF[split])

for split in ROLLF:
    print(f'{split}: {len(load_eps(split))} episodes')


## Residual policy and mean–CVaR objective


In [ ]:
# ── differentiable episode replay + BS residual policy + Rockafellar–Uryasev CVaR training ──
import torch, torch.nn as nn
DEVICE = torch.device('cpu')

def pack_episodes(eps_df):
    """flatten several episodes into flat tensors, for differentiable replay (state use the BS position trajectory, decoupled from the policy)."""
    states, b_pos, S_old, S_new, P_old, P_new, ep_id, is_first = [], [], [], [], [], [], [], []
    for ei, (_, erow) in enumerate(eps_df.iterrows()):
        df = erow['ep_step_df']; prev_b = 0.0
        for t in range(len(df)):
            r = df.iloc[t]
            states.append(build_state(r, prev_b))
            b = float(r['B Pos'])
            b_pos.append(b); S_old.append(float(r['S-1'])); S_new.append(float(r['S0']))
            P_old.append(float(r['P-1'])); P_new.append(float(r['P0']))
            ep_id.append(ei); is_first.append(1.0 if t == 0 else 0.0)
            prev_b = b
    t = lambda x, dt=torch.float32: torch.tensor(np.array(x), dtype=dt, device=DEVICE)
    return dict(states=t(states), b_pos=t(b_pos), S_old=t(S_old), S_new=t(S_new),
                P_old=t(P_old), P_new=t(P_new), ep_id=t(ep_id, torch.long),
                is_first=t(is_first), n_ep=int(eps_df.shape[0]))

class ResidualPolicy(nn.Module):
    def __init__(self, dim=8, hid=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(dim, hid), nn.ReLU(),
                                 nn.Linear(hid, hid), nn.ReLU(), nn.Linear(hid, 1))
        nn.init.zeros_(self.net[-1].weight); nn.init.zeros_(self.net[-1].bias)  # initial≈BS
    def forward(self, s): return self.net(s).squeeze(-1)

def _seg_sum(vals, ep_id, n_ep):
    out = torch.zeros(n_ep, device=vals.device, dtype=vals.dtype)
    return out.index_add_(0, ep_id, vals)

def _episode_reward(pack, pos, kappa):
    p_prev = torch.cat([torch.zeros(1, device=pos.device), pos[:-1]]) * (1.0 - pack['is_first'])
    tc  = -torch.abs(pos - p_prev) * pack['S_old'] * TC_RATE
    pnl = pos * (pack['S_new'] - pack['S_old']) + (pack['P_new'] - pack['P_old']) + tc
    q = pnl * 100.0
    step_r = (q - kappa * torch.relu(-q)) * 10.0          # new downside reward, differentiable (relu)
    return _seg_sum(step_r, pack['ep_id'], pack['n_ep'])

def _positions(pack, policy):
    adj = policy(pack['states'])
    return torch.clamp(pack['b_pos'] + adj, -1.5, 1.5)     # position = BS delta + residual (B Pos=-delta∈[-1,0])

def _train_residual_once(pack_tr, kappa, beta, alpha=0.05, epochs=400, lr=1e-3, seed=0,
                         init_std=0.01, warm_frac=0.25):
    """single training run: small random init of the last layer + pure mean for the first warm_frac(β=0) + β ramp linearly to the target (v9)."""
    torch.manual_seed(seed); np.random.seed(seed)
    policy = ResidualPolicy().to(DEVICE)
    torch.nn.init.normal_(policy.net[-1].weight, std=init_std)
    torch.nn.init.zeros_(policy.net[-1].bias)
    eta = torch.zeros(1, device=DEVICE, requires_grad=True)
    opt = torch.optim.Adam(list(policy.parameters()) + [eta], lr=lr)
    bs_r = _episode_reward(pack_tr, pack_tr['b_pos'], kappa).detach()
    for ep in range(epochs):
        bt = 0.0 if ep < warm_frac * epochs else \
             beta * min(1.0, (ep - warm_frac*epochs) / max(1.0, warm_frac*epochs))
        opt.zero_grad()
        delta = _episode_reward(pack_tr, _positions(pack_tr, policy), kappa) - bs_r
        cvar = eta - (1.0 / alpha) * torch.relu(eta - delta).mean()
        loss = -(delta.mean() + bt * cvar)
        loss.backward(); opt.step()
    return policy

def _adj_scale(policy, pack):
    with torch.no_grad():
        return policy(pack['states']).abs().mean().item()

COLLAPSE_TH, N_RETRY = 1e-3, 1   # collapse criterion(trainside) and restart count — report the trigger frequency in the paper's methods section

def train_residual(pack_tr, kappa, beta, alpha=0.05, epochs=400, lr=1e-3, seed=0, **kw):
    """v9 = warmup training + auto-restart on collapse (the criterion looks only at the train side |adj|, does not touch test; seed+1000 retry)."""
    policy = _train_residual_once(pack_tr, kappa, beta, alpha, epochs, lr, seed, **kw)
    a = _adj_scale(policy, pack_tr); tries = 0
    while a < COLLAPSE_TH and tries < N_RETRY:
        tries += 1
        print(f'  ↻ collapse-retry{tries}: κ={kappa} β={beta} seed {seed}→{seed+1000*tries} (|adj|={a:.5f})')
        p2 = _train_residual_once(pack_tr, kappa, beta, alpha, epochs, lr, seed + 1000*tries, **kw)
        a2 = _adj_scale(p2, pack_tr)
        if a2 > a: policy, a = p2, a2
    if a < COLLAPSE_TH:
        print(f'  ⚠ collapse persists: κ={kappa} β={beta} seed={seed} (|adj|={a:.5f})')
    return policy

@torch.no_grad()
def eval_delta(pack, policy, kappa):
    bs_r = _episode_reward(pack, pack['b_pos'], kappa)
    return (_episode_reward(pack, _positions(pack, policy), kappa) - bs_r).cpu().numpy()

@torch.no_grad()
def eval_rewards(pack, policy, kappa):
    r_pol = _episode_reward(pack, _positions(pack, policy), kappa).cpu().numpy()
    r_bs  = _episode_reward(pack, pack['b_pos'], kappa).cpu().numpy()
    return r_pol, r_bs

print('residual-CVaR machinery ready; device =', DEVICE)


## Training diagnostics


In [ ]:
import hashlib as _ahl, inspect as _ains
def _audit(tag, pack=None):
    parts = [f'[audit:{tag}]']
    try:
        parts.append('RATES=' + _ahl.md5(open('data/1yr_treasury.csv','rb').read()).hexdigest()[:8]
                     + f'({len(RATES)})')
    except Exception: parts.append('RATES=?')
    for f in (train_residual, _train_residual_once, _episode_reward, _positions, build_state):
        parts.append(f.__name__[:6] + '=' + _ahl.md5(_ains.getsource(f).encode()).hexdigest()[:6])
    if pack is not None:
        parts.append('states=' + _ahl.md5(pack['states'].detach().cpu().numpy().tobytes()).hexdigest()[:10])
    import torch as _t
    parts.append(f'thr={_t.get_num_threads()}')
    print(' '.join(parts))
print('audit ready')

## Paired estimates and block bootstrap


In [ ]:
import numpy as np

def _ep_dates(eps_df):
    """each episode's start date (datetime64, pack order)."""
    return pd.to_datetime(eps_df['date_start']).dt.floor('D').to_numpy()

def cvar5(a):
    a = np.asarray(a, float); k = max(1, int(0.05 * len(a)))
    return float(np.sort(a)[:k].mean())

def full_stat_ci(x, dates, stat=np.mean, B=3000, block_days=10, seed=0):
    """main convention: full-sample statistics + date-based blocks bootstrap CI.
    resampling unit=trading-day blocks (same-day episodes enter/exit as a whole), block length block_days absorbs temporal correlation."""
    x = np.asarray(x, float)
    d_uniq, inv = np.unique(dates, return_inverse=True)
    groups = [np.where(inv == k)[0] for k in range(len(d_uniq))]
    rng = np.random.default_rng(seed)
    point = float(stat(x))
    n = len(d_uniq); nb = int(np.ceil(n / block_days))
    bs = np.empty(B)
    for b in range(B):
        starts = rng.integers(0, n, nb)
        di = np.concatenate([(s + np.arange(block_days)) % n for s in starts])[:n]
        sel = np.concatenate([groups[k] for k in di])
        bs[b] = stat(x[sel])
    lo, hi = np.percentile(bs, [2.5, 97.5])
    return point, float(lo), float(hi), float((bs <= 0).mean())

def collapsed_spaced(x, dates, min_days=7):
    """auxiliary convention: same-day mean collapse → ≥min_days greedy spacing. Deterministic, no ties."""
    x = np.asarray(x, float)
    d_uniq, inv = np.unique(dates, return_inverse=True)
    daily = np.array([x[inv == k].mean() for k in range(len(d_uniq))])
    sel, last = [], None
    for k in range(len(d_uniq)):
        if last is None or (d_uniq[k] - last) / np.timedelta64(1, 'D') >= min_days:
            sel.append(k); last = d_uniq[k]
    sel = np.array(sel, dtype=int)
    return d_uniq[sel], daily[sel]

def spaced_mean_ci(x, dates, min_days=7, B=3000, L=5, seed=0):
    _, v = collapsed_spaced(x, dates, min_days)
    rng = np.random.default_rng(seed)
    n = len(v); nb = int(np.ceil(n / L))
    bs = np.empty(B)
    for b in range(B):
        starts = rng.integers(0, n, nb)
        idx = np.concatenate([(s + np.arange(L)) % n for s in starts])[:n]
        bs[b] = v[idx].mean()
    lo, hi = np.percentile(bs, [2.5, 97.5])
    return float(v.mean()), float(lo), float(hi), float((bs <= 0).mean()), n

def summarize_dp(dp, dates, tag=''):
    """one-line aggregate: main-convention trio + auxiliary convention. Returns a dict for the summary table."""
    m, lo, hi, p = full_stat_ci(dp, dates, np.mean)
    w, wl, wh, _ = full_stat_ci(dp, dates, lambda a: float((np.asarray(a) > 0).mean()))
    t, tl, th, _ = full_stat_ci(dp, dates, cvar5)
    sm, sl, sh, sp_, nsp = spaced_mean_ci(dp, dates)
    print(f'{tag} full sample: ΔPnL={m:+.2f}bp [{lo:+.2f},{hi:+.2f}] p_le0={p:.3f} | '
          f'win={w:.0%} [{wl:.0%},{wh:.0%}] | tailCVaR={t:+.2f} [{tl:+.2f},{th:+.2f}]')
    print(f'{tag} collapse by day×≥7days (n={nsp}): ΔPnL={sm:+.2f}bp [{sl:+.2f},{sh:+.2f}] p_le0={sp_:.3f}')
    return dict(full=m, full_lo=lo, full_hi=hi, full_p=p, win=w, win_lo=wl, win_hi=wh,
                tail=t, tail_lo=tl, tail_hi=th, sp=sm, sp_lo=sl, sp_hi=sh, sp_p=sp_,
                n_spaced=nsp)

print('✓ E0 unified estimator ready (full_stat_ci / collapsed_spaced / summarize_dp)')

## Regime configuration


In [ ]:
# Switch the data split and load its validation-selected beta.
_REGIME_PRESETS = {
    'crash_test2018': (list(range(2008, 2017)), [2017], [2018]),
    'crash_test2020': (list(range(2008, 2019)), [2019], [2020]),
    'mix_test2022':   (list(range(2008, 2021)), [2021], [2022]),
    'mix_test2024':   (list(range(2008, 2023)), [2023], [2024]),
}

def switch_regime(regime):
    """switch the global context to the specified regime: years/split/cache incorporation/ROLLF/fingerprint validation/β*load."""
    global REGIME, TRAIN_YEARS, VAL_YEARS, TEST_YEARS, CRASH_SLICES, OUT_DIR, ROLLF
    global CFG_TAG, _SPLIT_YEARS, RATES, best_beta
    global ALPHA, EPOCHS, LR, KAPPAS, POLICY_SEEDS
    import hashlib, glob, shutil, json as _json

    ty, vy, te_y = _REGIME_PRESETS[regime]
    assert max(ty) < min(vy) and max(vy) < min(te_y), 'LOOK-AHEAD!'
    REGIME, TRAIN_YEARS, VAL_YEARS, TEST_YEARS = regime, ty, vy, te_y
    CRASH_SLICES = CRASH_WINDOWS.get(regime, [])
    OUT_DIR = f'{DRIVE_REPO}/checkpoints/regime_{regime}'
    os.makedirs(OUT_DIR, exist_ok=True)
    print(f'━━ switch → {regime}: train {ty[0]}-{ty[-1]} | val {vy} | test {te_y} '
          f'| crashslice={bool(CRASH_SLICES)}')

    # 1) rewrite data/{train,validation,test}.csv + Treasury rate (and ## 2 same logic)
    df_all = pd.read_parquet(CLEAN_PARQUET)
    df_all['date'] = pd.to_datetime(df_all['date'])
    yr = df_all['date'].dt.year
    out_cols = ['quote_datetime', 'ticker', 'expiration', 'strike',
                'underlying_bid', 'underlying_ask', 'bid', 'ask', 'nbr_next_steps']
    for name, years in [('train', ty), ('validation', vy), ('test', te_y)]:
        sub = df_all.loc[yr.isin(years)].sort_values(
            ['expiration', 'strike', 'date']).reset_index(drop=True)
        g = sub.groupby(['expiration', 'strike'], sort=False)
        sub['nbr_next_steps'] = (g['date'].transform('count') - g.cumcount() - 1).astype('int32')
        sub[out_cols].to_csv(f'data/{name}.csv', index=False)
    rates = (df_all[['date', 'r']].drop_duplicates('date').sort_values('date')
             .rename(columns={'date': 'Date', 'r': '1y'}))
    rates['Date'] = pd.to_datetime(rates['Date']).dt.strftime('%Y-%m-%d')
    old = pd.read_csv('data/1yr_treasury.csv') if os.path.exists('data/1yr_treasury.csv') else pd.DataFrame()
    pd.concat([old, rates]).drop_duplicates('Date').sort_values('Date').to_csv(
        'data/1yr_treasury.csv', index=False)
    RATES = pd.read_csv('data/1yr_treasury.csv'); RATES['Date'] = pd.to_datetime(RATES['Date'])
    del df_all

    # 2) recomputed for this regime CFG_TAG / _SPLIT_YEARS (fixes the batch-phase tag freeze issue)
    CFG_TAG = hashlib.md5(repr((sorted(TRAIN_YEARS), sorted(VAL_YEARS), sorted(TEST_YEARS),
                                REP_NAME)).encode()).hexdigest()[:8]
    _SPLIT_YEARS = {'train': set(map(int, TRAIN_YEARS)),
                    'validation': set(map(int, VAL_YEARS)),
                    'test': set(map(int, TEST_YEARS))}

    # 4) ROLLF + hard validation + fingerprint archived
    ROLLF = {sp: gen_rollout(sp) for sp in ['train', 'validation', 'test']}
    import torch as _th
    print(f'[env] torch={_th.__version__}  '
          f'dev={_th.cuda.get_device_name(0) if _th.cuda.is_available() else "CPU"}')
    print(f'[fp ] parquet md5={_file_md5(CLEAN_PARQUET)}')
    for sp in ['train', 'validation', 'test']:
        yrs = _csv_years(ROLLF[sp]); exp = _SPLIT_YEARS[sp]
        assert yrs <= exp, f'{sp}: year out of range {sorted(yrs - exp)}!'
        assert not (exp - yrs), f'{sp}: missing years {sorted(exp - yrs)}!'
        print(f'[fp ] {sp:11s} {min(yrs)}–{max(yrs)} ✓  md5={_file_md5(ROLLF[sp])}')

    # 5) constants + β* (reads the batch-phase on-disk json, does not rerun the search)
    ALPHA, EPOCHS, LR = 0.05, 400, 1e-3
    KAPPAS, POLICY_SEEDS = [1, 2, 3], [0, 1, 2, 3, 4]
    with open(f'{OUT_DIR}/regime_best_beta.json') as f:
        best_beta = {int(k): float(v) for k, v in _json.load(f).items()}
    print(f'★ β* = {best_beta}   (from {OUT_DIR}/regime_best_beta.json)')
    print('Regime ready:', REGIME)

print('switch_regime() ready — usage: switch_regime("mix_test2022")')


## Robustness experiments

The breach-target sweep uses epsilon in {0.05, 0.10, 0.20}. Transaction costs
are 5, 10 and 20 bp. The cost experiment supports both evaluation of fixed
policies and optional retraining (`DO_TC_RETRAIN`).

This notebook records the original diagnostics. The paper's cost and turnover
summaries are read by the reproduction script from the saved experiment
records; see [the result map](../docs/reproduction.md#paper-results).


### R0. Select a regime


In [ ]:
PANEL_REGIME = 'mix_test2022'      # change this line + restart the kernel to extend to other regime
switch_regime(PANEL_REGIME)
import pandas as pd, numpy as np, torch
ALPHA, EPOCHS, LR = 0.05, 400, 1e-3
KAPPAS, POLICY_SEEDS = [1, 2, 3], [0, 1, 2, 3, 4]
EPS_TR, EPS_VA, EPS_TE = load_eps('train'), load_eps('validation'), load_eps('test')
pack_tr, pack_va, pack_te = pack_episodes(EPS_TR), pack_episodes(EPS_VA), pack_episodes(EPS_TE)
BEST_BETA_BASE = dict(best_beta)
DATES_TE = _ep_dates(EPS_TE)
SP_TE = np.array([float(e['ep_step_df']['S-1'].iloc[0]) for _, e in EPS_TE.iterrows()])

def _episode_pnl(pack, pos):
    p_prev = torch.cat([torch.zeros(1, device=pos.device), pos[:-1]]) * (1.0 - pack['is_first'])
    tc  = -torch.abs(pos - p_prev) * pack['S_old'] * TC_RATE
    pnl = pos * (pack['S_new'] - pack['S_old']) + (pack['P_new'] - pack['P_old']) + tc
    return _seg_sum(pnl, pack['ep_id'], pack['n_ep'])
BS_PNL_TE = _episode_pnl(pack_te, pack_te['b_pos']).detach().cpu().numpy()

EXP_DIR = f'{DRIVE_REPO}/checkpoints/robust_checks_{PANEL_REGIME}'; os.makedirs(EXP_DIR, exist_ok=True)
print('R0 ready |', PANEL_REGIME, '| β* =', BEST_BETA_BASE, '| TC_RATE =', TC_RATE)


### Deployment rules


In [ ]:
# ── G2. extended CRC mechanism: v3 closed-loop mechanism + four-control rules (fixv / valstatic / emp / ivrule) ──
EXP_NAME = 'conformal_controls'
EXP_DIR  = f'{DRIVE_REPO}/checkpoints/{EXP_NAME}_{REGIME}'; os.makedirs(EXP_DIR, exist_ok=True)
LAM_GRID = [0.0, 0.25, 0.5, 0.75, 1.0]
EPS_T    = 0.10

def _episode_pnl(pack, pos):
    p_prev = torch.cat([torch.zeros(1, device=pos.device), pos[:-1]]) * (1.0 - pack['is_first'])
    tc  = -torch.abs(pos - p_prev) * pack['S_old'] * TC_RATE
    pnl = pos * (pack['S_new'] - pack['S_old']) + (pack['P_new'] - pack['P_old']) + tc
    return _seg_sum(pnl, pack['ep_id'], pack['n_ep'])

@torch.no_grad()
def episode_delta_lambda(pack, policy, kappa, lam_vec):
    """reward convention Δ (for evaluation, consistent with the v7 report)."""
    adj = policy(pack['states'])
    lam_step = torch.as_tensor(np.asarray(lam_vec, dtype=np.float32), device=DEVICE)[pack['ep_id']]
    pos = torch.clamp(pack['b_pos'] + lam_step * adj, -1.5, 1.5)
    r_pol = _episode_reward(pack, pos, kappa)
    r_bs  = _episode_reward(pack, pack['b_pos'], kappa)
    return (r_pol - r_bs).cpu().numpy(), r_pol.cpu().numpy(), r_bs.cpu().numpy()

@torch.no_grad()
def episode_dpnl_bp_lambda(pack, policy, lam_vec, start_spot):
    """PnL convention Δ, in bp of notional (for control/budget, κ-unrelated)."""
    adj = policy(pack['states'])
    lam_step = torch.as_tensor(np.asarray(lam_vec, dtype=np.float32), device=DEVICE)[pack['ep_id']]
    pos = torch.clamp(pack['b_pos'] + lam_step * adj, -1.5, 1.5)
    dp = (_episode_pnl(pack, pos) - _episode_pnl(pack, pack['b_pos'])).cpu().numpy()
    return dp / start_spot * 1e4

def conformal_lower_q(v, w, eps):
    """weighted conformal lower tail ε quantile, virtual point −∞ weight 1 (conservative correction). Insufficient samples → −inf."""
    v = np.asarray(v, float); w = np.asarray(w, float)
    W = w.sum() + 1.0
    if W <= 0 or 1.0 / W >= eps: return -np.inf
    idx = np.argsort(v); v, w = v[idx], w[idx]
    cw = (1.0 + np.cumsum(w)) / W          # −∞ mass entirely at the left end
    k = int(np.searchsorted(cw, eps, side='left'))
    return float(v[min(k, len(v) - 1)])

def precompute_grids(eps_df, pack, policy, kappa):
    """each (policy, split) computed only once: date/spot/IV + all λ the grid's reward-Δ and PnL-Δbp."""
    n  = len(eps_df)
    ds = pd.to_datetime(eps_df['date_start']).to_numpy()
    de = np.array([pd.to_datetime(e['ep_step_df']['Date']).max()
                   for _, e in eps_df.iterrows()], dtype='datetime64[ns]')
    ss = np.array([float(e['ep_step_df']['S-1'].iloc[0]) for _, e in eps_df.iterrows()])
    iv = np.array([float(e['ep_step_df']['v'].iloc[0])  for _, e in eps_df.iterrows()])
    order = np.argsort(ds)
    d_lam  = {la: episode_delta_lambda(pack, policy, kappa, np.full(n, la))[0] for la in LAM_GRID}
    dp_lam = {la: episode_dpnl_bp_lambda(pack, policy, np.full(n, la), ss)     for la in LAM_GRID}
    return dict(n=n, ds=ds, de=de, ss=ss, iv=iv, order=order, d_lam=d_lam, dp_lam=dp_lam)

def calibrate_budget_bp(pack_va, eps_va_df, policy, eps_target, floor_bp=5.0):
    """budget c = validation fix1's ΔPnL_bp of ε absolute quantile value (lower bound floor_bp)."""
    ss = np.array([float(e['ep_step_df']['S-1'].iloc[0]) for _, e in eps_va_df.iterrows()])
    dp = episode_dpnl_bp_lambda(pack_va, policy, np.ones(len(eps_va_df)), ss)
    return float(max(floor_bp, -np.quantile(dp, eps_target)))

def deploy_lambda_control(eps_df, pack, policy, kappa, mode='crc_rwc', eps_target=0.10,
                          budget_bp=25.0, grids=None, gamma_eps=0.02, half_life=30.0,
                          tau_iv=0.04, warmup=15, M_emp=0.3, lam_warmup=1.0, lam_fix=None):
    """chronological episode-by-episode deployment λ_t.
       mode ∈ {fix1, fix0, fixv(λ=lam_fix), a2margin, emp, ivrule, aci_open, crc, crc_rwc, crc_rwc_aci}"""
    g = grids if grids is not None else precompute_grids(eps_df, pack, policy, kappa)
    n, ds, de, iv, order = g['n'], g['ds'], g['de'], g['iv'], g['order']
    d_lam, dp_lam = g['d_lam'], g['dp_lam']
    used = np.full(n, np.nan); informed = np.zeros(n, bool)
    lam_aci = float(lam_warmup); eps_eff = float(eps_target)
    aci_pending = []                                   # crc_rwc_aci delayed-feedback queue
    for t, i in enumerate(order):
        past = order[:t]
        done = past[de[past] < ds[i]]                  # ★ use only completed episode (fix look-ahead)
        informed[i] = len(done) >= warmup
        # —— crc_rwc_aci: first digest"completed by now"deployment results, online-tune ε_eff (closed-loop + delayed feedback) ——
        if mode == 'crc_rwc_aci' and aci_pending:
            still = []
            for j in aci_pending:
                if de[j] < ds[i]:
                    err = float(dp_lam[used[j]][j] < -budget_bp)
                    eps_eff = float(np.clip(eps_eff + gamma_eps * (eps_target - err),
                                            0.25 * eps_target, min(0.45, 4 * eps_target)))
                else:
                    still.append(j)
            aci_pending = still
        # —— decision ——
        if   mode == 'fix1': lam_t = 1.0
        elif mode == 'fix0': lam_t = 0.0
        elif mode == 'fixv': lam_t = float(lam_fix)    # controls one/two: any fixed λ (with valstatic)
        elif not informed[i]:
            lam_t = 0.0 if mode == 'a2margin' else lam_warmup
        elif mode == 'a2margin':                       # old A2-style (completion filtering fixed)
            thr = cvar_5(d_lam[0.0][done]) - M_emp
            lam_t = 0.0
            for la in sorted(LAM_GRID, reverse=True):
                if cvar_5(d_lam[la][done]) >= thr: lam_t = la; break
        elif mode == 'emp':                            # control three: rolling empirical breach-rate rule (no weighting/no correction)
            lam_t = 0.0
            for la in sorted(LAM_GRID, reverse=True):
                if float(np.mean(dp_lam[la][done] < -budget_bp)) <= eps_target:
                    lam_t = la; break
        elif mode == 'ivrule':                         # control four: IV quantile haircut (thresholds 50/80 preregistered)
            p_iv = float(np.mean(iv[done] <= iv[i]))
            lam_t = 1.0 if p_iv < 0.5 else (0.5 if p_iv < 0.8 else 0.25)
        elif mode == 'aci_open':                       # old open-loop rule (ablation control, only fixes completion filtering)
            var_hat = np.quantile(dp_lam[1.0][done], eps_target)
            err = float(dp_lam[1.0][i] < var_hat)      # events remain aligned with deployment λ unrelated → open-loop
            lam_aci = float(np.clip(lam_aci + 0.10 * (eps_target - err), 0.0, 1.0))
            lam_t = lam_aci
        else:                                          # crc / crc_rwc / crc_rwc_aci (closed-loop)
            if mode in ('crc_rwc', 'crc_rwc_aci'):
                dt_ = (ds[i] - de[done]) / np.timedelta64(1, 'D')
                w = (np.exp(-np.log(2) * np.maximum(dt_, 0.0) / half_life)
                     * np.exp(-np.abs(iv[done] - iv[i]) / tau_iv))
            else:
                w = np.ones(len(done))
            e_use = eps_eff if mode == 'crc_rwc_aci' else eps_target
            lam_t = 0.0
            for la in sorted(LAM_GRID, reverse=True):  # risk varies with λ monotonically ⇒ take the largest feasible λ
                if conformal_lower_q(dp_lam[la][done], w, e_use) >= -budget_bp:
                    lam_t = la; break
        used[i] = lam_t
        if mode == 'crc_rwc_aci': aci_pending.append(i)
    # —— unified evaluation: reward basis + post-deployment actual breach (all modes comparable) ——
    d_used, rp, rbs = episode_delta_lambda(pack, policy, kappa, used)
    dp_used = episode_dpnl_bp_lambda(pack, policy, used, g['ss'])
    m = informed
    breach = float(np.mean(dp_used[m] < -budget_bp)) if m.any() else np.nan
    return {'lam': used, 'd': d_used, 'rp': rp, 'rbs': rbs, 'dpnl_bp': dp_used,
            'breach': breach, 'informed': m, 'budget_bp': float(budget_bp),
            'eps_eff_final': float(eps_eff)}

def lam_val_static(policy, kappa, budget_bp, eps_target=EPS_T):
    """control two: validation year, take max{λ: experience breach ≤ ε} (λ=0 always feasible ⇒ always has a solution)."""
    gva = precompute_grids(EPS_VA, pack_va, policy, kappa)
    for la in sorted(LAM_GRID, reverse=True):
        if float(np.mean(gva['dp_lam'][la] < -budget_bp)) <= eps_target:
            return float(la)
    return 0.0

def deploy_mode(mode, pol, kappa, sd, grids, lamval, budget):
    """unified entry point: fix25/50/75 and valstatic go through fixv, the rest native mode."""
    if mode in ('fix25', 'fix50', 'fix75'):
        return deploy_lambda_control(EPS_TE, pack_te, pol, kappa, mode='fixv',
                                     lam_fix=float(mode[3:]) / 100.0, eps_target=EPS_T,
                                     budget_bp=budget, grids=grids)
    if mode == 'valstatic':
        return deploy_lambda_control(EPS_TE, pack_te, pol, kappa, mode='fixv',
                                     lam_fix=lamval[(kappa, sd)], eps_target=EPS_T,
                                     budget_bp=budget, grids=grids)
    return deploy_lambda_control(EPS_TE, pack_te, pol, kappa, mode=mode,
                                 eps_target=EPS_T, budget_bp=budget, grids=grids)

ALL_MODES = ['fix0', 'fix25', 'fix50', 'fix75', 'fix1', 'valstatic', 'ivrule', 'emp',
             'a2margin', 'aci_open', 'crc', 'crc_rwc', 'crc_rwc_aci']
V3_SIX    = ['fix1', 'a2margin', 'aci_open', 'crc', 'crc_rwc', 'crc_rwc_aci']
V3_ARM5   = ['fix1', 'aci_open', 'crc', 'crc_rwc', 'crc_rwc_aci']

import hashlib as _h, inspect as _i
for _fn in [deploy_lambda_control, conformal_lower_q, precompute_grids,
            calibrate_budget_bp, lam_val_static, deploy_mode]:
    print('[src]', f'{_fn.__name__:22s}', _h.md5(_i.getsource(_fn).encode()).hexdigest()[:8])
print('extended CRC mechanism ready — 13 rule | EXP_DIR =', EXP_DIR)


### R1. Prepare the validation-selected and aggressive policies


In [ ]:
BASE_POL, WILD_POL = {}, {}
for kappa in KAPPAS:
    for sd in POLICY_SEEDS:
        BASE_POL[(kappa, sd)] = train_residual(pack_tr, kappa, BEST_BETA_BASE[kappa], ALPHA, EPOCHS, LR, seed=sd)
        WILD_POL[(kappa, sd)] = train_residual(pack_tr, kappa, 0.0, ALPHA, EPOCHS, LR, seed=sd)
print('trained base + wild (15 each)')


### R2. Breach-target sensitivity


In [ ]:
# ε∈{5,10,20}%: each ε recalibrate budget c and redeploy crc_rwc_aci (referenced against fix1).
EPS_GRID = [0.05, 0.10, 0.20]
rows = []
for eps_t in EPS_GRID:
    for kappa in KAPPAS:
        budg = {sd: calibrate_budget_bp(pack_va, EPS_VA, WILD_POL[(kappa, sd)], eps_t) for sd in POLICY_SEEDS}
        for mode in ['fix1', 'crc_rwc_aci']:
            paths, br = [], []
            for sd in POLICY_SEEDS:
                g = precompute_grids(EPS_TE, pack_te, WILD_POL[(kappa, sd)], kappa)
                out = deploy_lambda_control(EPS_TE, pack_te, WILD_POL[(kappa, sd)], kappa,
                                            mode=mode, eps_target=eps_t,
                                            budget_bp=budg[sd], grids=g)
                paths.append(out['dpnl_bp']); br.append(out['breach'])
            dpm = np.mean(paths, axis=0)
            m, lo, hi, p = full_stat_ci(dpm, DATES_TE, np.mean)
            t, _, _, _ = full_stat_ci(dpm, DATES_TE, cvar5)
            rows.append(dict(eps=eps_t, kappa=kappa, mode=mode, mean=m, lo=lo, hi=hi,
                             breach=float(np.nanmean(br)), tail=t))
            print(f'ε={eps_t:.2f} κ={kappa} {mode:12s}: mean={m:+.2f} [{lo:+.2f},{hi:+.2f}] '
                  f'breach={np.nanmean(br):.3f} tail={t:+.2f}')
EPS_DF = pd.DataFrame(rows); EPS_DF.to_csv(f'{EXP_DIR}/eps_sweep_{PANEL_REGIME}.csv', index=False)
print('saved →', f'{EXP_DIR}/eps_sweep_{PANEL_REGIME}.csv')
print('expected: ε the smaller → breach the lower it is, the less mean is retained; ε the larger → conversely (the knob is monotonic).')


### R3. Turnover and trade frequency


In [ ]:
# per-episode notional turnover (bp) and trade count; also store Δturnover(policy−BS) for TC recomputation.
def _turn_dollar(pack, pos):
    p_prev = torch.cat([torch.zeros(1, device=pos.device), pos[:-1]]) * (1.0 - pack['is_first'])
    return _seg_sum(torch.abs(pos - p_prev) * pack['S_old'], pack['ep_id'], pack['n_ep'])
def _ntrades(pack, pos):
    p_prev = torch.cat([torch.zeros(1, device=pos.device), pos[:-1]]) * (1.0 - pack['is_first'])
    return _seg_sum((torch.abs(pos - p_prev) > 1e-6).float(), pack['ep_id'], pack['n_ep'])

bs_turn_d = _turn_dollar(pack_te, pack_te['b_pos']).detach().cpu().numpy()      # $ per episode
DTURN_BP = {}                                                                   # Δturnover (bp) per episode, base arm
rows = []
for kappa in KAPPAS:
    base_turn = np.mean([_turn_dollar(pack_te, _positions(pack_te, BASE_POL[(kappa, sd)])).detach().cpu().numpy()
                         for sd in POLICY_SEEDS], axis=0)
    wild_turn = np.mean([_turn_dollar(pack_te, _positions(pack_te, WILD_POL[(kappa, sd)])).detach().cpu().numpy()
                         for sd in POLICY_SEEDS], axis=0)
    base_ntr  = np.mean([_ntrades(pack_te, _positions(pack_te, BASE_POL[(kappa, sd)])).detach().cpu().numpy()
                         for sd in POLICY_SEEDS], axis=0)
    DTURN_BP[kappa] = (base_turn - bs_turn_d) / SP_TE * 1e4
    rows.append(dict(kappa=kappa,
                     turn_bs_bp=float(np.mean(bs_turn_d / SP_TE * 1e4)),
                     turn_base_bp=float(np.mean(base_turn / SP_TE * 1e4)),
                     turn_wild_bp=float(np.mean(wild_turn / SP_TE * 1e4)),
                     d_turn_base_bp=float(np.mean(DTURN_BP[kappa])),
                     trades_per_ep_base=float(np.mean(base_ntr))))
    print(f'κ={kappa}: turnover(bp/ep) BS={rows[-1]["turn_bs_bp"]:.0f} base={rows[-1]["turn_base_bp"]:.0f} '
          f'wild={rows[-1]["turn_wild_bp"]:.0f} | Δturn(base−BS)={rows[-1]["d_turn_base_bp"]:+.1f}bp '
          f'| trades/ep≈{rows[-1]["trades_per_ep_base"]:.1f}')
TURN_DF = pd.DataFrame(rows); TURN_DF.to_csv(f'{EXP_DIR}/turnover_{PANEL_REGIME}.csv', index=False)
print('saved →', f'{EXP_DIR}/turnover_{PANEL_REGIME}.csv')


### R4. Transaction-cost sensitivity


In [ ]:
# lower-bound convention: the base policy trained at 10bp, analytically, at TC∈{5,10,20}bp recompute the edge.
# ΔPnL(tc) = ΔPnL(10bp) − (tc − 0.001)·Δturnover_bp   (the same policy, true·retraining can only be better ⇒ this is a lower bound)
TC_GRID = [0.0005, 0.001, 0.002]
rows = []
for kappa in KAPPAS:
    dp10 = np.mean([(_episode_pnl(pack_te, _positions(pack_te, BASE_POL[(kappa, sd)])).detach().cpu().numpy()
                     - BS_PNL_TE) / SP_TE * 1e4 for sd in POLICY_SEEDS], axis=0)
    for tc in TC_GRID:
        dp_tc = dp10 - (tc - 0.001) * DTURN_BP[kappa]
        m, lo, hi, p = full_stat_ci(dp_tc, DATES_TE, np.mean)
        rows.append(dict(kappa=kappa, tc_bp=tc*1e4, kind='recost_lowerbound',
                         mean=m, lo=lo, hi=hi, p_le0=p))
        print(f'[recost] κ={kappa} TC={tc*1e4:.0f}bp: mean={m:+.2f} [{lo:+.2f},{hi:+.2f}] p≤0={p:.3f}')

DO_TC_RETRAIN = False    # ← set True for a full retrain (TC into the training objective; per tier, retrain 15 base policies, ≈1–1.5h/tiers)
if DO_TC_RETRAIN:
    # train_residual cost is computed from the module-global TC_RATE; change globals()['TC_RATE'] then retrain for it to take effect.
    for tc in [0.0005, 0.002]:
        globals()['TC_RATE'] = tc
        bs_pnl = _episode_pnl(pack_te, pack_te['b_pos']).detach().cpu().numpy()
        for kappa in KAPPAS:
            dps = []
            for sd in POLICY_SEEDS:
                pol = train_residual(pack_tr, kappa, BEST_BETA_BASE[kappa], ALPHA, EPOCHS, LR, seed=sd)
                dps.append((_episode_pnl(pack_te, _positions(pack_te, pol)).detach().cpu().numpy()
                            - bs_pnl) / SP_TE * 1e4)
            m, lo, hi, p = full_stat_ci(np.mean(dps, axis=0), DATES_TE, np.mean)
            rows.append(dict(kappa=kappa, tc_bp=tc*1e4, kind='full_retrain', mean=m, lo=lo, hi=hi, p_le0=p))
            print(f'[retrain] κ={kappa} TC={tc*1e4:.0f}bp: mean={m:+.2f} [{lo:+.2f},{hi:+.2f}] p≤0={p:.3f}')
    globals()['TC_RATE'] = 0.001   # restore
TC_DF = pd.DataFrame(rows); TC_DF.to_csv(f'{EXP_DIR}/tc_sweep_{PANEL_REGIME}.csv', index=False)
print('saved →', f'{EXP_DIR}/tc_sweep_{PANEL_REGIME}.csv')
print('criterion: if the lower bound at 20bp still edge>0 (p≤0 small) ⇒ no retraining needed, "edge in 2× survives under the cost"; otherwise kick off DO_TC_RETRAIN.')


### R5. Save results


In [ ]:
import json as _jj, time as _tt
_man = {'created': _tt.strftime('%Y-%m-%d %H:%M:%S'), 'regime': PANEL_REGIME,
        'env': {'torch': torch.__version__, 'threads': torch.get_num_threads()},
        'eps_grid': EPS_GRID, 'tc_grid_bp': [t*1e4 for t in TC_GRID],
        'do_tc_retrain': bool(DO_TC_RETRAIN),
        'files': ['eps_sweep_%s.csv' % PANEL_REGIME, 'turnover_%s.csv' % PANEL_REGIME,
                  'tc_sweep_%s.csv' % PANEL_REGIME],
        'note': 'tail-α and block-bootstrap by robust_reanalysis.py computed separately(zero compute, saved to disk perep_dp).'}
with open(f'{EXP_DIR}/robust_manifest_{PANEL_REGIME}.json', 'w') as fh:
    _jj.dump(_man, fh, ensure_ascii=False, indent=1)
print('★ Robustness ③ done →', EXP_DIR)
